<a href="https://colab.research.google.com/github/ManviGumber08/comp3132/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Name : Manvi Gumber
#Student Id : 101412099

In [8]:
# Import required libraries
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import cifar10
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [9]:
#Load and prepare CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # normalize

# Create validation set
x_val = x_train[-10000:]
y_val = y_train[-10000:]
x_train = x_train[:-10000]
y_train = y_train[:-10000]


PART 2: Build a model with Overfitting


In [10]:
#Build a simple ConvNet (deliberately overfit)
model_overfit = models.Sequential([
    layers.Conv2D(256, (3, 3), activation='relu', input_shape=(32, 32, 3)),  # Overparameterized
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_overfit.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Train and plot results
history_overfit = model_overfit.fit(x_train, y_train, epochs=10,
                                    validation_data=(x_val, y_val))

# Plot loss & accuracy
plt.plot(history_overfit.history['loss'], label='Train Loss')
plt.plot(history_overfit.history['val_loss'], label='Val Loss')
plt.title('Overfitting: Loss')
plt.legend()
plt.show()

plt.plot(history_overfit.history['accuracy'], label='Train Accuracy')
plt.plot(history_overfit.history['val_accuracy'], label='Val Accuracy')
plt.title('Overfitting: Accuracy')
plt.legend()
plt.show()

PART 3: Apply Techniques to Reduce Overfitting

In [11]:
#Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
datagen.fit(x_train)


In [12]:
#Rebuild the model with Dropout + L2 Regularization
model_regularized = models.Sequential([
    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(0.001), input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),
    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_regularized.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])


In [ ]:
#Train the improved model
history_reg = model_regularized.fit(datagen.flow(x_train, y_train, batch_size=64),
                                    epochs=10,  # More epochs with regularization
                                    validation_data=(x_val, y_val))


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 102s 164ms/step - accuracy: 0.3413 - loss: 1.8883 - val_accuracy: 0.4736 - val_loss: 1.5667
Epoch 2/10
207/625 ━━━━━━━━━━━━━━━━━━━━ 1:03 152ms/step - accuracy: 0.4209 - loss: 1.7068

In [ ]:
#Plot updated results
plt.plot(history_reg.history['loss'], label='Train Loss')
plt.plot(history_reg.history['val_loss'], label='Val Loss')
plt.title('Regularized: Loss')
plt.legend()
plt.show()

plt.plot(history_reg.history['accuracy'], label='Train Accuracy')
plt.plot(history_reg.history['val_accuracy'], label='Val Accuracy')
plt.title('Regularized: Accuracy')
plt.legend()
plt.show()


PART 4: Explanations (Markdown in Colab/Jupyter)
## Data Preparation
We used CIFAR-10 dataset and normalized it between 0 and 1. We split it into training, validation, and test sets.

## Overfitting Model
We trained a model with too many parameters and observed a gap between training and validation accuracy — a classic sign of overfitting.

## Overfitting Reduction Techniques
We used:
1. Dropout layers to randomly deactivate neurons.
2. L2 Regularization to penalize large weights.
3. Data Augmentation to increase data diversity.

## Evaluation
We trained the model for 30 epochs. The regularized model showed much smaller loss and better generalization on validation data.
